In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

import joblib

# Configuration des graphiques pour un rendu professionnel
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10
# 1️⃣ CHARGEMENT DES DONNÉES
df = pd.read_csv("USA_Housing.csv")
df.columns = df.columns.str.lower().str.replace(" ", "_")

print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")
print(f"Valeurs nulles : {df.isnull().sum().sum()}")



Dataset chargé : 4856 lignes, 7 colonnes
Valeurs nulles : 0


In [7]:
# 2️⃣ FEATURE ENGINEERING (SIMPLE & JUSTIFIÉ)
df['rooms_per_bedroom'] = df['avg_area_number_of_rooms'] / df['avg_area_number_of_bedrooms']
df['income_per_capita'] = df['avg_area_income'] / df['area_population']
df['density_score'] = df['area_population'] / df['avg_area_number_of_rooms']


In [8]:
# 3️⃣ SÉLECTION DES FEATURES
features = [
    'avg_area_income',
    'avg_area_house_age',
    'avg_area_number_of_rooms',
    'avg_area_number_of_bedrooms',
    'area_population',
    'rooms_per_bedroom',
    'income_per_capita',
    'density_score'
]

X = df[features]
y = df['price']

In [9]:
# 4️⃣ SPLIT TRAIN / TEST (AVANT suppression des outliers pour éviter data leakage)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [10]:

# 5️⃣ SUPPRESSION DES OUTLIERS (SEULEMENT sur le train pour éviter data leakage)
# ------------------------------
Q1_train = y_train.quantile(0.25)
Q3_train = y_train.quantile(0.75)
IQR_train = Q3_train - Q1_train
lower_bound_train = Q1_train - 1.5 * IQR_train
upper_bound_train = Q3_train + 1.5 * IQR_train

# Filtrer seulement le train
train_mask = (y_train >= lower_bound_train) & (y_train <= upper_bound_train)
X_train = X_train[train_mask]
y_train = y_train[train_mask]
print(f"Après suppression des outliers : {len(X_train)} lignes dans le train (sur {len(X_test)} dans le test)")

Après suppression des outliers : 3884 lignes dans le train (sur 972 dans le test)


In [12]:
# 6️⃣ MODÈLES DE BASE
base_models = [
    ('ridge', Ridge()),
    ('rf', RandomForestRegressor(random_state=42)),
    ('gb', GradientBoostingRegressor(random_state=42))
]


In [13]:
# 7️⃣ MODÈLE STACKING


stacking = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(alpha=1.0),  # régularisation → stabilise le stacking
    cv=3,
    n_jobs=-1
)

# 8️⃣ PIPELINE AVEC ColumnTransformer
numeric_features = features  # toutes les features sont numériques

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features)
    ],
    remainder='passthrough'  # inutile ici mais correct
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', stacking)
])

In [14]:
# 9️⃣ GRID SEARCH (OPTIMISATION RÉELLE)
param_grid = {
    'model__ridge__alpha': [0.1, 1, 10, 50],

    'model__rf__n_estimators': [100, 200],
    'model__rf__max_depth': [10, 15, 20],

    'model__gb__learning_rate': [0.01, 0.05, 0.1],
    'model__gb__n_estimators': [100, 150, 200],

    'model__final_estimator__alpha': [0.1, 1, 10, 50]
}

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='r2',
    n_jobs=-1
)

In [15]:
# 🔟 FIT
grid.fit(X_train, y_train)

# 1️⃣1️⃣ ÉVALUATION FINALE
best_model = grid.best_estimator_

y_pred = best_model.predict(X_test)

r2 = r2_score(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)

print("Meilleur modèle (Stacking optimisé)")
print(f"Meilleurs paramètres: {grid.best_params_}")
print(f"\n=== PERFORMANCES SUR TEST SET ===")
print(f"R²   : {r2:.4f}")
print(f"RMSE : {rmse:,.2f}")
print(f"MAE  : {mae:,.2f}")

# Comparaison train/test pour détecter overfitting
y_train_pred = best_model.predict(X_train)
r2_train = r2_score(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae_train = mean_absolute_error(y_train, y_train_pred)
ecart_r2 = r2_train - r2
print(f"\n=== PERFORMANCES SUR TRAIN SET ===")
print(f"R²   : {r2_train:.4f}")
print(f"RMSE : {rmse_train:,.2f}")
print(f"MAE  : {mae_train:,.2f}")
overfitting_msg = "⚠️ Overfitting possible" if ecart_r2 > 0.1 else "✅ Pas d'overfitting"
print(f"\nÉcart R² (train-test): {ecart_r2:.4f} {overfitting_msg}")


Meilleur modèle (Stacking optimisé)
Meilleurs paramètres: {'model__final_estimator__alpha': 50, 'model__gb__learning_rate': 0.05, 'model__gb__n_estimators': 100, 'model__rf__max_depth': 15, 'model__rf__n_estimators': 200, 'model__ridge__alpha': 1}

=== PERFORMANCES SUR TEST SET ===
R²   : 0.9045
RMSE : 100,829.17
MAE  : 81,609.20

=== PERFORMANCES SUR TRAIN SET ===
R²   : 0.9058
RMSE : 104,171.80
MAE  : 83,742.89

Écart R² (train-test): 0.0013 ✅ Pas d'overfitting


In [16]:
# 1️⃣2️⃣ VISUALISATIONS COMPLÈTES POUR PFE

# ========== GRAPHIQUE 1 : Prédictions vs Valeurs réelles ==========
plt.figure(figsize=(10, 8))
plt.scatter(y_test, y_pred, alpha=0.6, s=50)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', lw=2, label='Ligne parfaite (y=x)')
plt.xlabel("Prix réel ($)", fontsize=12, fontweight='bold')
plt.ylabel("Prix prédit ($)", fontsize=12, fontweight='bold')
plt.title(f"Prédictions vs Valeurs réelles\nR² = {r2:.4f} | RMSE = {rmse:,.0f}$", fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('01_predictions_vs_reality.png', dpi=300, bbox_inches='tight')
plt.close()

# ========== GRAPHIQUE 2 : Distribution des résidus ==========
residuals = y_test - y_pred
plt.figure(figsize=(10, 6))
plt.hist(residuals, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
plt.axvline(x=0, color='r', linestyle='--', linewidth=2, label='Erreur nulle')
plt.xlabel("Résidus (Prix réel - Prix prédit) ($)", fontsize=12, fontweight='bold')
plt.ylabel("Fréquence", fontsize=12, fontweight='bold')
plt.title(f"Distribution des Résidus\nMoyenne = {residuals.mean():,.0f}$ | Écart-type = {residuals.std():,.0f}$", 
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('02_distribution_residus.png', dpi=300, bbox_inches='tight')
plt.close()

# ========== GRAPHIQUE 3 : Résidus vs Prédictions (détection de patterns) ==========
plt.figure(figsize=(10, 6))
plt.scatter(y_pred, residuals, alpha=0.6, s=50)
plt.axhline(y=0, color='r', linestyle='--', linewidth=2, label='Erreur nulle')
plt.xlabel("Prix prédit ($)", fontsize=12, fontweight='bold')
plt.ylabel("Résidus ($)", fontsize=12, fontweight='bold')
plt.title("Résidus vs Prédictions\n(Pour détecter des patterns dans les erreurs)", 
          fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('03_residus_vs_predictions.png', dpi=300, bbox_inches='tight')
plt.close()

# ========== GRAPHIQUE 4 : Distribution des prix (réels vs prédits) ==========
plt.figure(figsize=(12, 6))
plt.hist(y_test, bins=50, alpha=0.6, label='Prix réels', color='blue', edgecolor='black')
plt.hist(y_pred, bins=50, alpha=0.6, label='Prix prédits', color='orange', edgecolor='black')
plt.xlabel("Prix ($)", fontsize=12, fontweight='bold')
plt.ylabel("Fréquence", fontsize=12, fontweight='bold')
plt.title("Distribution des Prix : Réels vs Prédits", fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('04_distribution_prix.png', dpi=300, bbox_inches='tight')
plt.close()

# ========== GRAPHIQUE 5 : Comparaison Train vs Test ==========
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Train
axes[0].scatter(y_train, y_train_pred, alpha=0.6, s=50, color='green')
axes[0].plot([y_train.min(), y_train.max()],
             [y_train.min(), y_train.max()], 'r--', lw=2)
axes[0].set_xlabel("Prix réel ($)", fontsize=11, fontweight='bold')
axes[0].set_ylabel("Prix prédit ($)", fontsize=11, fontweight='bold')
axes[0].set_title(f"Train Set\nR² = {r2_train:.4f}", fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Test
axes[1].scatter(y_test, y_pred, alpha=0.6, s=50, color='blue')
axes[1].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()], 'r--', lw=2)
axes[1].set_xlabel("Prix réel ($)", fontsize=11, fontweight='bold')
axes[1].set_ylabel("Prix prédit ($)", fontsize=11, fontweight='bold')
axes[1].set_title(f"Test Set\nR² = {r2:.4f}", fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.suptitle("Comparaison des Performances : Train vs Test", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('05_train_vs_test.png', dpi=300, bbox_inches='tight')
plt.close()

# ========== GRAPHIQUE 6 : Matrice de corrélation des features ==========
plt.figure(figsize=(10, 8))
correlation_matrix = X_train.corr()
mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', 
            center=0, square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title("Matrice de Corrélation des Features", fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('06_correlation_matrix.png', dpi=300, bbox_inches='tight')
plt.close()

# ========== GRAPHIQUE 7 : Importance des features (via RandomForest) ==========
# Extraire le RandomForest du pipeline pour obtenir l'importance des features
rf_model = best_model.named_steps['model'].named_estimators_['rf']
feature_importance = rf_model.feature_importances_
feature_names = features

# Créer un DataFrame pour faciliter le tri
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importance
}).sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='steelblue', edgecolor='black')
plt.xlabel("Importance", fontsize=12, fontweight='bold')
plt.ylabel("Features", fontsize=12, fontweight='bold')
plt.title("Importance des Features (Random Forest)", fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig('07_feature_importance.png', dpi=300, bbox_inches='tight')
plt.close()

# ========== GRAPHIQUE 8 : Comparaison des modèles de base ==========
# Évaluer chaque modèle de base individuellement
base_model_scores = {}
for name, model in base_models:
    # Créer un pipeline simple pour chaque modèle
    simple_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    simple_pipeline.fit(X_train, y_train)
    y_pred_base = simple_pipeline.predict(X_test)
    r2_base = r2_score(y_test, y_pred_base)
    base_model_scores[name] = r2_base

# Ajouter le modèle stacking
base_model_scores['Stacking'] = r2

# Créer le graphique
models = list(base_model_scores.keys())
scores = list(base_model_scores.values())
colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

plt.figure(figsize=(10, 6))
bars = plt.bar(models, scores, color=colors, edgecolor='black', linewidth=1.5)
plt.ylabel("R² Score", fontsize=12, fontweight='bold')
plt.xlabel("Modèles", fontsize=12, fontweight='bold')
plt.title("Comparaison des Performances des Modèles", fontsize=14, fontweight='bold')
plt.ylim([min(scores) - 0.05, max(scores) + 0.05])
plt.grid(True, alpha=0.3, axis='y')

# Ajouter les valeurs sur les barres
for bar, score in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{score:.4f}', ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.savefig('08_comparaison_modeles.png', dpi=300, bbox_inches='tight')
plt.close()

print("\n✅ Tous les graphiques ont été générés et sauvegardés :")
print("   01_predictions_vs_reality.png")
print("   02_distribution_residus.png")
print("   03_residus_vs_predictions.png")
print("   04_distribution_prix.png")
print("   05_train_vs_test.png")
print("   06_correlation_matrix.png")
print("   07_feature_importance.png")
print("   08_comparaison_modeles.png")

# 1️⃣3️⃣ SAUVEGARDE DU MODÈLE
joblib.dump(best_model, 'housing_model.joblib')
print("Modèle sauvegardé sous 'housing_model.joblib'")




✅ Tous les graphiques ont été générés et sauvegardés :
   01_predictions_vs_reality.png
   02_distribution_residus.png
   03_residus_vs_predictions.png
   04_distribution_prix.png
   05_train_vs_test.png
   06_correlation_matrix.png
   07_feature_importance.png
   08_comparaison_modeles.png
Modèle sauvegardé sous 'housing_model.joblib'
